<a href="https://colab.research.google.com/github/Franncippi/Proyecto-mentoria-M09/blob/Fran/notebooks/03_modelado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctico 3 — Modelado

**Mentoría M09 — El impacto de la IA en la fuerza laboral**

Dataset: [diplodatos-ai-impact-workforce](https://github.com/carolinapepe/diplodatos-ai-impact-workforce)

---

## Qué hace esta notebook

El Práctico 1 dejó dos hipótesis planteadas y el Práctico 2 preparó los datos para responderlas, dejando además un catálogo de variables por rol y una serie de advertencias puntuales para el momento de modelar. Esta notebook retoma la **Hipótesis 2** — ¿es posible predecir la pérdida o creación de empleo a partir de las características de adopción de IA de una empresa? — y aplica sobre ella un enfoque mixto: clasificación supervisada como método principal, y clustering no supervisado como evidencia complementaria e independiente.

## Resumen de decisiones (para leer antes que el código)

1. **El target no es `saldo_neto_empleo`.** Tiene fiabilidad 0,33 (P2, §10.6.2): dos tercios de su varianza es ruido de medición, y al estar centrado en cero, ese ruido cae justo sobre el borde de decisión de un clasificador binario ingenuo. En su lugar modelamos `tasa_creacion` (fiabilidad 0,90) y `tasa_desplazamiento` (fiabilidad 0,79) como dos clasificadores binarios independientes, cada uno partido por su mediana.
2. **El clustering nunca ve los resultados.** Se arma únicamente con variables de rol *predictor* (estructura, adopción, operación, gobernanza, país). Los resultados (`tasa_creacion`, `tasa_desplazamiento`, `productivity_change_percent`, etc.) se calculan *después*, por clúster, solo para describir — nunca para agrupar.
3. **Unidad de análisis: la tabla por empresa**, no el panel. El panel exige `GroupKFold` por `company_id` y el P2 mostró que su dimensión temporal es, para este target, ruido blanco (§10.6.1) — no se gana señal real usando las 150.000 filas del panel en vez de las 10.000 empresas.
4. **`task_automation_rate` se reporta con y sin.** Está a 0,91 de correlación con la adopción y conceptualmente pegada al mecanismo de desplazamiento (P2, §11.2): la incluimos porque es la hipótesis a testear, pero mostramos el efecto de sacarla.
5. **El clustering es complementario, no un reemplazo.** Sirve para triangular: si los perfiles de alta adopción muestran, en promedio, mejor balance de creación/desplazamiento que los de baja adopción, tenemos dos métodos independientes apuntando en la misma dirección — evidencia más fuerte que un solo modelo por sí solo.

---
# 1. Enfoque elegido

## 1.1 La pregunta que venimos a responder (P1 → H2)

La **Hipótesis 2** del Práctico 1 planteaba que es posible predecir la pérdida o creación de empleo de una empresa a partir de sus características de adopción de IA, proponiendo originalmente `ai_use_case` (el tipo de tarea automatizada) como mecanismo central — con el argumento de que *qué* se automatiza importa más que *cuánto*.

El Práctico 2 (§2.5, §2.6) debilitó esa formulación puntual: `ai_use_case` resultó estar determinada por la industria (cada industria sortea entre un menú fijo de 4 casos de uso) y no aporta información propia de la empresa una vez que `industry` ya está en el modelo. Mantenemos la pregunta de fondo de la H2 — *¿la intensidad y la forma de adopción de IA predicen el resultado laboral de la empresa?* — pero la respondemos con el conjunto completo de predictores de adopción, estructura y gobernanza que sí mostraron estructura real (P2, §11.2), en lugar de apoyarnos en una única variable categórica sin señal propia.

## 1.2 Por qué un enfoque mixto: supervisado + no supervisado

La pregunta tiene dos lecturas legítimas, y cada una pide una técnica distinta:

| Lectura de la H2 | Pregunta que responde | Técnica |
|---|---|---|
| ¿Se puede **predecir** el resultado de empleo de una empresa a partir de cómo adoptó la IA? | Predicción puntual | **Supervisado** (clasificación) |
| ¿Existen **perfiles** de empresa adoptadora que se comporten distinto frente al empleo, más allá de lo que capture una sola variable a la vez? | Estructura latente | **No supervisado** (clustering) |

No son alternativas — son complementarias. El enfoque supervisado da una respuesta directa y cuantificable a "se puede predecir, con tal desempeño". El no supervisado no predice nada: agrupa empresas por similitud en su forma de adoptar IA sin mirar nunca el resultado, y solo después compara el resultado promedio entre grupos. Si las dos vías coinciden — los perfiles de adopción más intensiva muestran, en promedio, mejor balance de creación/desplazamiento, y el clasificador logra separar esos mismos casos — la evidencia a favor de la H2 es más sólida que la que daría cualquiera de las dos por separado, porque son dos formas de ataque metodológicamente independientes sobre la misma pregunta.

## 1.3 Enfoque supervisado — dos clasificadores binarios

Descartamos clasificar el signo de `saldo_neto_empleo` por el motivo ya mencionado: es la resta de dos cantidades con fiabilidad alta, y restar cancela señal y suma ruido (P2, registro de decisiones, punto 6). En su lugar entrenamos **dos clasificadores binarios independientes**, cada uno sobre una tasa con fiabilidad alta partida por su mediana:

| Target | Definición | Fiabilidad (P2 §10.6.2) | Corte |
|---|---|---|---|
| `alta_creacion` | 1 si `tasa_creacion` ≥ mediana, 0 si no | 0,90 | Mediana → balance 50/50 por construcción |
| `alto_desplazamiento` | 1 si `tasa_desplazamiento` ≥ mediana, 0 si no | 0,79 | Mediana → balance 50/50 por construcción |

Los predictores en ambos casos son los 25 del catálogo `PREDICTORES` de P2 §11.2 (estructura, adopción, operación, gobernanza, contexto país), excluyendo siempre las variables de rol *resultado* (serían fuga de información/circularidad, no predictores contemporáneos). `task_automation_rate` se incluye pero se reporta también un modelo sin ella, por su cercanía conceptual al mecanismo de desplazamiento.

Partir por la mediana en vez de por cero tiene una ventaja adicional sobre clasificar el saldo neto: el balance de clases queda 50/50 *por construcción*, así que no hace falta ninguna técnica de manejo de desbalance — el punto 2 del práctico (preparación final) se simplifica en ese aspecto.

## 1.4 Enfoque no supervisado — perfiles de adopción (clustering)

> **Regla de separación (P2, §11.2):** si se clusteriza incluyendo variables de resultado, los grupos se separan por resultado — y después "descubrir" que el grupo que más adoptó IA es el que mejor le fue es circular. El clustering se arma solo con predictores; los resultados se miran después, para describir.

Usamos las variables numéricas de los mismos cinco bloques de predictores (`PRED_ESTRUCTURA`, `PRED_ADOPCION`, `PRED_OPERACION`, `PRED_GOBERNANZA`, `PRED_PAIS`) para agrupar a las empresas por similitud, estandarizando antes de calcular distancias. Las categóricas (`industry`, `region`, `ai_adoption_stage`, etc.) quedan afuera del clustering propiamente dicho y se usan, junto con `tasa_creacion` y `tasa_desplazamiento`, para describir cada clúster una vez formado — nunca para formarlo.

La técnica (K-means o alternativa) y el número de clusters se deciden en la sección de implementación, con un criterio cuantitativo (silhouette / codo) combinado con que los grupos resulten interpretables en términos de negocio.

## 1.5 Ventajas del enfoque

- Ataca el problema real de fiabilidad en vez de ignorarlo: modela las dos variables más confiables del dataset en lugar de su resta ruidosa.
- El corte por mediana da balance de clases perfecto, sin necesidad de sobremuestreo/submuestreo.
- Permite un hallazgo más rico que un solo número de accuracy: si los predictores de "alta creación" y "alto desplazamiento" resultan distintos, sugiere que crear y destruir empleo son mecanismos separados, no dos caras de la misma moneda — algo que el saldo neto, al fusionarlos en una resta, no podría mostrar nunca.
- El clustering aporta una segunda fuente de evidencia, metodológicamente independiente del clasificador, para la misma pregunta (triangulación).
- Ninguno de los dos métodos usa variables de rol *resultado* como predictoras, así que ninguno corre el riesgo de un modelo tautológico.

## 1.6 Limitaciones y riesgos identificados

- **Sigue habiendo un techo de fiabilidad**, aunque más alto que con el saldo neto (0,90 y 0,79 en vez de 0,33): parte de la varianza de cada tasa sigue siendo ruido trimestral, así que ningún clasificador va a acercarse a un accuracy perfecto, y no debería esperarse eso.
- **Partir por la mediana es relativo, no absoluto**: una empresa justo por encima y otra justo por debajo de la mediana pueden tener tasas de creación casi idénticas. El corte separa "la mitad superior" de la distribución, no necesariamente dos poblaciones cualitativamente distintas.
- **`task_automation_rate` sigue siendo un caso límite** conceptualmente cercano al mecanismo que se quiere probar; reportar el modelo con y sin ella mitiga el riesgo pero no lo elimina del todo.
- **El clustering no prueba causalidad.** Si un clúster de alta adopción muestra mejor balance de empleo, es una asociación observacional entre perfiles — no evidencia de que la adopción de IA *cause* ese resultado.
- **El dataset es sintético y generado por reglas** (P2, conclusiones finales): cualquier patrón que encuentren los dos métodos puede ser un artefacto del generador y no un fenómeno real del mercado laboral. Esto no invalida el ejercicio, pero limita qué tan fuerte puede ser el lenguaje de las conclusiones.

---
# 2. Preparación final

Esta sección cubre primero la preparación específica del **enfoque no supervisado** (clustering), que es autosuficiente y no depende de ningún target. La preparación del enfoque supervisado (definición de los dos targets binarios, split, etc.) se agrega más adelante en esta misma sección, cuando pasemos a esa parte.

## 2.1 Carga de datos

El bloque siguiente intenta primero la ruta relativa local (`../data/processed/`), pensada para cuando esta notebook corre dentro del repo clonado (Jupyter local). Si no la encuentra —caso típico de Colab, que no clona el repo solo por abrir el `.ipynb`— clona el repositorio en la propia sesión y ajusta la ruta. Así la notebook corre igual en los dos entornos sin tocar código.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

SEMILLA = 42
np.random.seed(SEMILLA)

RUTA_LOCAL = Path("../data/processed/empresas.parquet")
if RUTA_LOCAL.exists():
    ruta_datos = RUTA_LOCAL
else:
    # Colab no trae el repo solo por abrir la notebook: lo clonamos nosotros
    !git clone -b modelado-tp3 --depth 1 https://github.com/Franncippi/Proyecto-mentoria-M09.git repo_m09
    ruta_datos = Path("repo_m09/data/processed/empresas.parquet")

empresas = pd.read_parquet(ruta_datos)
print(empresas.shape)
print("nulos:", empresas.isna().sum().sum())

## 2.2 Selección de variables para el clustering

Usamos únicamente las variables numéricas de rol **predictor** (P2, §11.2): estructura, adopción, operación, gobernanza y contexto país. Quedan afuera a propósito:

- Las **categóricas** (`industry`, `region`, `ai_adoption_stage`, `country_ai_policy`, `ai_ethics_committee`): K-means trabaja con distancia euclídea y no las maneja de forma nativa. Se reservan para describir los clusters después, no para formarlos.
- Las **variables de resultado** (`tasa_creacion`, `tasa_desplazamiento`, `saldo_neto_empleo`, `productivity_change_percent`, etc.): si entraran acá, los clusters se separarían por resultado, y cualquier lectura posterior sería circular (P2, §11.2). Se usan exclusivamente después, para describir cada cluster.
- `company_id`: identificador, nunca predictor.

In [ ]:
PRED_ESTRUCTURA = ["num_employees", "annual_revenue_usd_millions", "company_age"]
PRED_ADOPCION   = ["ai_adoption_rate", "years_using_ai", "num_ai_tools_used",
                    "ai_projects_active", "ai_budget_percentage", "ai_training_hours",
                    "antiguedad_ia_relativa"]
PRED_OPERACION  = ["task_automation_rate", "remote_work_percentage"]
PRED_GOBERNANZA = ["ai_failure_rate", "regulatory_compliance_score", "ai_risk_management_score"]
PRED_PAIS       = ["gdp_per_capita", "internet_penetration", "digital_maturity_index",
                    "ai_patent_filings_2024", "ai_researchers_per_million"]

NUM_CLUSTER = PRED_ESTRUCTURA + PRED_ADOPCION + PRED_OPERACION + PRED_GOBERNANZA + PRED_PAIS
print(f"{len(NUM_CLUSTER)} variables numericas para clustering")

empresas[NUM_CLUSTER].describe().T[["mean", "std", "min", "max"]].round(2)

## 2.3 Escalado

Las escalas son muy dispares (`num_employees` con desvío ≈ 4.760, `ai_budget_percentage` con desvío ≈ 2,5): sin estandarizar, la distancia euclídea de K-means quedaría dominada por las variables de mayor magnitud, sin relación con cuánta información aportan. Aplicamos `StandardScaler` (media 0, desvío 1) antes de cualquier cálculo de distancia — obligatorio acá, a diferencia de un modelo basado en árboles.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(empresas[NUM_CLUSTER])

print("media tras escalar (deberia ser ~0):", X_scaled.mean(axis=0).round(2)[:5])
print("desvio tras escalar (deberia ser ~1):", X_scaled.std(axis=0).round(2)[:5])

## 2.4 Reducción de dimensionalidad con PCA

Justificación (ver también sección 1.4):

1. **Colinealidad real entre las variables de adopción** — lo confirmamos abajo: varias correlacionan entre 0,74 y 0,94. Sin corregirlo, esa dimensión conceptual pesaría varias veces en la distancia.
2. **Reduce la maldición de la dimensionalidad** propia de los métodos basados en distancia con ~20 variables (Beyer et al., 1999).
3. **Permite interpretar y graficar** los perfiles en 2-3 ejes en vez de 20.

El orden es `StandardScaler` → `PCA` → `KMeans`, nunca al revés.

In [ ]:
corr_adopcion = empresas[PRED_ADOPCION + ["task_automation_rate"]].corr().round(2)
corr_adopcion

In [ ]:
pca_full = PCA(random_state=SEMILLA)
pca_full.fit(X_scaled)

var_exp = pca_full.explained_variance_ratio_
cum_var = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, len(var_exp)+1), var_exp, "o-")
axes[0].set(title="Scree plot", xlabel="Componente", ylabel="Varianza explicada")
axes[1].plot(range(1, len(cum_var)+1), cum_var, "o-", color="seagreen")
axes[1].axhline(0.80, color="gray", linestyle="--", label="80%")
axes[1].set(title="Varianza acumulada", xlabel="Componente", ylabel="Acumulada")
axes[1].legend()
plt.tight_layout()
plt.show()

n80 = int(np.argmax(cum_var >= 0.80) + 1)
n_kaiser = int((pca_full.explained_variance_ > 1).sum())
print(f"Componentes para 80% de varianza: {n80}")
print(f"Componentes con autovalor > 1 (criterio de Kaiser): {n_kaiser}")

**Decisión: nos quedamos con 5 componentes** (≈84% de la varianza total), un poco por encima del criterio de Kaiser (4) para no perder la variable de tamaño de empresa, que aparece recién en PC4.

In [ ]:
N_COMPONENTES = 5
pca = PCA(n_components=N_COMPONENTES, random_state=SEMILLA)
X_pca = pca.fit_transform(X_scaled)
print("Varianza retenida:", pca.explained_variance_ratio_.sum().round(3))

loadings = pd.DataFrame(pca.components_[:4].T, index=NUM_CLUSTER,
                         columns=["PC1", "PC2", "PC3", "PC4"])
loadings.reindex(loadings.PC1.abs().sort_values(ascending=False).index).round(2)

**Lectura de los componentes** (mirando qué variables originales pesan más en cada uno):

- **PC1 — intensidad/madurez de adopción de IA**: pesa positivo en `ai_training_hours`, `ai_adoption_rate`, `ai_budget_percentage`, `ai_projects_active`, `task_automation_rate`, `num_ai_tools_used`, `ai_risk_management_score`; negativo en `ai_failure_rate`. Es, en una sola dimensión, "qué tan a fondo adoptó la IA la empresa, y con qué tan poca fricción".
- **PC2 — contexto macro del país**: `gdp_per_capita`, `ai_researchers_per_million`, `digital_maturity_index`, `internet_penetration`, `regulatory_compliance_score`. Es todo lo que depende de dónde está la empresa, no de cómo es.
- **PC3 — antigüedad relativa de la adopción**: `company_age` (positivo) y `antiguedad_ia_relativa` (negativo) — separa empresas viejas que adoptaron IA hace relativamente poco de empresas jóvenes que la tienen desde que existen.
- **PC4 — tamaño de empresa**: `annual_revenue_usd_millions` y `num_employees`.

Que la estructura tenga una lectura de negocio tan limpia (y que coincida con los bloques con los que arrancamos: adopción, país, antigüedad, estructura) es una señal de que el PCA no está mezclando cosas sin sentido.

---
# 3. Implementación y evaluación — enfoque no supervisado

## 3.1 Elección de k

Usamos K-means sobre los 5 componentes, comparando inercia (codo) y silhouette score para varios valores de k.

In [ ]:
resultados_k = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    resultados_k.append({"k": k, "inercia": km.inertia_, "silhouette": sil})

tabla_k = pd.DataFrame(resultados_k)
tabla_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tabla_k.k, tabla_k.inercia, "o-")
axes[0].set(title="Metodo del codo", xlabel="k", ylabel="Inercia")
axes[1].plot(tabla_k.k, tabla_k.silhouette, "o-", color="darkorange")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
plt.show()

## 3.2 Comparación de configuraciones (k y con/sin `task_automation_rate`)

El silhouette no baja monótonamente con k — hay que mirar además si los clusters adicionales aportan algo interpretable. Comparamos k=2, 3 y 4 mirando el resultado (`tasa_creacion`, `tasa_desplazamiento`, `saldo_neto_empleo`) **solo a modo descriptivo, después de formar los grupos** — nunca se usó para agruparlos.

In [ ]:
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10)
    empresas[f"cluster_k{k}"] = km.fit_predict(X_pca)
    print(f"--- k={k} ---")
    resumen = empresas.groupby(f"cluster_k{k}")[
        ["ai_adoption_rate", "tasa_creacion", "tasa_desplazamiento", "saldo_neto_empleo"]
    ].mean().round(2)
    resumen["n_empresas"] = empresas[f"cluster_k{k}"].value_counts().sort_index()
    print(resumen, "\n")

In [ ]:
# Robustez frente a task_automation_rate: comparamos el clustering final con y sin esa variable
SIN_TASK_AUTO = [v for v in NUM_CLUSTER if v != "task_automation_rate"]
X_scaled_sin = StandardScaler().fit_transform(empresas[SIN_TASK_AUTO])
X_pca_sin = PCA(n_components=N_COMPONENTES, random_state=SEMILLA).fit_transform(X_scaled_sin)

labels_con = empresas["cluster_k2"]
labels_sin = KMeans(n_clusters=2, random_state=SEMILLA, n_init=10).fit_predict(X_pca_sin)

ari = adjusted_rand_score(labels_con, labels_sin)
print(f"Adjusted Rand Index (con vs. sin task_automation_rate): {ari:.3f}")

**Elección final: k=2.** Empata en el mejor silhouette junto con k=3, pero con k=3 o k=4 los clusters adicionales terminan subdividiendo al grupo de baja adopción en subgrupos con comportamiento casi idéntico (mismo `tasa_creacion`/`tasa_desplazamiento` promedio) — no aportan una lectura nueva, así que por parsimonia nos quedamos con la solución más simple. El resultado además es robusto a sacar `task_automation_rate` (ARI alto entre ambas versiones): no depende de esa variable puntual.

In [ ]:
cluster_final = empresas["cluster_k2"]
empresas["perfil_adopcion"] = cluster_final.map({0: "baja_adopcion", 1: "alta_adopcion"})

empresas.groupby("perfil_adopcion")[
    ["ai_adoption_rate", "ai_risk_management_score",
     "tasa_creacion", "tasa_desplazamiento", "saldo_neto_empleo"]
].mean().round(2)